In [ ]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine



In [2]:
env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:"
    f"{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:"
    f"{os.getenv('DB_PORT')}/"
    f"{os.getenv('DB_NAME')}"
)

In [ ]:
orders = pd.read_sql("SELECT * FROM orders;", engine)
order_items = pd.read_sql("SELECT * FROM order_items;", engine)
products = pd.read_sql("SELECT * FROM products;", engine)

print( orders.shape)
print(order_items.shape)
print(products.shape)

Orders: (30033, 9)
Order Items: (78610, 7)
Products: (208, 10)


In [4]:
transaction_data = (
    order_items
    .merge(
        orders[["order_id", "customer_id"]],
        on="order_id",
        how="inner"
    )
    .merge(
        products[["product_id", "product_name"]],
        on="product_id",
        how="inner"
    )
)

transaction_data.head()


,order_item_id,order_id,product_id,quantity,unit_price,discount,item_total,customer_id,product_name
0,OI00000001,ORD0000001,PROD0171,2,330.0,32.5,445.50,CUST001444,Scholastic Picture Book S2
1,OI00000002,ORD0000001,PROD0177,3,4120.0,6.0,11618.40,CUST001444,Hot Wheels Block Set 100
2,OI00000003,ORD0000001,PROD0156,1,470.0,13.6,406.08,CUST001444,Westland Storybook 200
3,OI00000004,ORD0000001,PROD0159,1,280.0,18.1,229.32,CUST001444,HarperCollins Bestseller Fiction Pro
4,OI00000005,ORD0000002,PROD0044,4,3150.0,6.9,11730.60,CUST003248,Van Heusen Slim Jeans 500


In [5]:
print(transaction_data.shape)
print(transaction_data[
    ["order_id", "customer_id", "product_id", "product_name"]
].head(10))

(78610, 9)
     order_id customer_id product_id                           product_name
0  ORD0000001  CUST001444   PROD0171             Scholastic Picture Book S2
1  ORD0000001  CUST001444   PROD0177               Hot Wheels Block Set 100
2  ORD0000001  CUST001444   PROD0156                 Westland Storybook 200
3  ORD0000001  CUST001444   PROD0159   HarperCollins Bestseller Fiction Pro
4  ORD0000002  CUST003248   PROD0044              Van Heusen Slim Jeans 500
5  ORD0000002  CUST003248   PROD0142               Nivia Tennis Racquet 100
6  ORD0000003  CUST000269   PROD0010                      Noise Pavilion S2
7  ORD0000003  CUST000269   PROD0007                    LG Smart LED TV Pro
8  ORD0000003  CUST000269   PROD0203  American Tourister Travel Duffel Plus
9  ORD0000004  CUST000948   PROD0129                     Amul Biscuits 500g


In [ ]:
transactions = (transaction_data.groupby("order_id")["product_name"].apply(list).tolist())
transactions[:5]
print(len(transactions))
print(transactions[0])

Number of transactions: 30033
First transaction: ['Scholastic Picture Book S2', 'Hot Wheels Block Set 100', 'Westland Storybook 200', 'HarperCollins Bestseller Fiction Pro']


In [9]:
from mlxtend.preprocessing import TransactionEncoder

encoder = TransactionEncoder()

basket_array = encoder.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    basket_array,
    columns=encoder.columns_
)

basket.head()
print(basket.shape)
print(basket.stack().unique())

(30033, 208)
[False  True]


In [ ]:
product_freq=basket.sum().sort_values(ascending=False)
product_freq.head(15)
print(product_freq.max())
print(product_freq.min())
print("products purchased in at least 1% of orders:-",(product_freq >=len(basket)*0.01).sum() )

LEGO Superhero Toy 400                      927
Hasbro Superhero Toy 200                    896
boAt Galaxy 200                             747
Dell Note 100                               704
American Tourister Chronograph Watch 400    692
Titan Analog Watch S2                       654
American Tourister Analog Watch 200         648
Decathlon Track Suit 400                    640
Adidas Running Shoes Plus                   632
Yonex Running Shoes 100                     631
Yonex Track Suit 200                        629
Bloomsbury Self-Help Guide 500              614
Penguin Biography Max                       611
Nivia Track Suit X1                         610
Decathlon Track Suit Plus                   607
dtype: int64

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

freq_itemsets=fpgrowth(basket, min_support=0.002,use_colnames=True)
freq_itemsets = freq_itemsets.sort_values("support", ascending=False)
freq_itemsets.head(15)

,support,itemsets
85,0.030866,frozenset({LEGO Superhero Toy 400})
164,0.029834,frozenset({Hasbro Superhero Toy 200})
87,0.024873,frozenset({boAt Galaxy 200})
41,0.023441,frozenset({Dell Note 100})
22,0.023041,frozenset({American Tourister Chronograph Watc...
54,0.021776,frozenset({Titan Analog Watch S2})
37,0.021576,frozenset({American Tourister Analog Watch 200})
146,0.021310,frozenset({Decathlon Track Suit 400})
134,0.021044,frozenset({Adidas Running Shoes Plus})
170,0.021010,frozenset({Yonex Running Shoes 100})


In [19]:
print(len(freq_itemsets))
freq_itemsets["itemset_size"]=(freq_itemsets["itemsets"].apply(len))
freq_itemsets["itemset_size"].value_counts().sort_index()

303


itemset_size
1    208
2     93
3      2
Name: count, dtype: int64

In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    freq_itemsets.drop(columns="itemset_size"),
    metric="confidence",
    min_threshold=0.1
)

rules = rules.sort_values("lift", ascending=False)

rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(15)

In [ ]:
useful_rules=rules[
    (rules["confidence"] >=0.20) &
    (rules["lift"] > 2) &
    (rules["support"] > 0.002)
].copy()

useful_rules = useful_rules.sort_values(["lift","confidence"], ascending=False)
print(len(rules))
print(len(useful_rules))
useful_rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(15)

In [30]:
#fallback
popular_products = (basket.sum().sort_values(ascending=False))
popular_products.head(10)

LEGO Superhero Toy 400                      927
Hasbro Superhero Toy 200                    896
boAt Galaxy 200                             747
Dell Note 100                               704
American Tourister Chronograph Watch 400    692
Titan Analog Watch S2                       654
American Tourister Analog Watch 200         648
Decathlon Track Suit 400                    640
Adidas Running Shoes Plus                   632
Yonex Running Shoes 100                     631
dtype: int64

In [55]:
def recommend_products(purchased_products,rules_df, popular_products,top_n=5):
    purchased=set(purchased_products)
    recommendations={}

    for _, row in rules_df.iterrows():
        antecedents = set(row["antecedents"])
        consequents= set(row["consequents"])

        if antecedents.issubset(purchased):
            for product in consequents:
                if product not in purchased:
                    if product not in recommendations:
                        recommendations[product]={
                            "confidence":row["confidence"],
                            "lift":row["lift"],
                            "support":row["support"],
                            "source":"Association Rule"
                        }

    ranked = sorted(
        recommendations.items(),
        key=lambda x : (x[1]["lift"], x[1]["confidence"],x[1]["support"]),
        reverse=True
    )


    results = [
        {
            "recommended_product":product,
            **metrics
        }
        for product, metrics in ranked[:top_n]
    ]

    recommended={item["recommended_product"] for item in results }

    for product in popular_products.index:
        if len(results) >= top_n:
            break
        if product not in purchased and product not in recommended:
            results.append({
                "recommended_product" : product,
                "confidence":None,
                "lift":None,
                "support":None,
                "source":"Popularity Fallback"
            })

            recommended.add(product)
    return pd.DataFrame(results)



In [56]:
customer_id = "CUST001444"

customer_purchases = pd.read_sql(
    """
    SELECT DISTINCT p.product_name
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.customer_id = %(customer_id)s
      AND o.order_status = 'Delivered';
    """,
    engine,
    params={"customer_id": customer_id}
)

purchased_products = customer_purchases["product_name"].tolist()

print("Customer:", customer_id)
print("Previously purchased products:", purchased_products)

Customer: CUST001444
Previously purchased products: ['boAt Galaxy 200', 'Cello Table Lamp 300', 'Fastrack Formal Belt X1', 'HarperCollins Bestseller Fiction Lite', 'HarperCollins Bestseller Fiction Pro', 'Hot Wheels Block Set 100', 'HP Over-Ear Headphone Lite', 'JBL Pulse Smartwatch X1', 'Mi Inspiron Plus', 'Mi Over-Ear Headphone Max', 'Mi Wireless Earbuds Plus', 'Nivea Matte Lipstick 200', 'Noise Pavilion S2', 'Penguin Biography Max', 'Rupa Novel Vol 500', 'Scholastic Picture Book S2', 'The Body Shop Body Mist Plusml', 'Westland Biography Max', 'Westland Storybook 200']


In [57]:
recommendations = recommend_products(
    purchased_products=purchased_products,
    rules_df=useful_rules,
    popular_products=popular_products,
    top_n=5
)

print(recommendations.columns.tolist())
display(recommendations)

['recommended_product', 'confidence', 'lift', 'support', 'source']


,recommended_product,confidence,lift,support,source
0,Bloomsbury Self-Help Guide 500,0.613139,29.990870,0.002797,Association Rule
1,Dell Note 100,0.318608,13.591970,0.007925,Association Rule
2,LEGO Superhero Toy 400,0.205882,6.670188,0.003729,Association Rule
3,Hasbro Superhero Toy 200,NaN,NaN,NaN,Popularity Fallback
4,American Tourister Chronograph Watch 400,NaN,NaN,NaN,Popularity Fallback


In [59]:
# Prepare order dates for evaluation
order_dates = orders[
    ["order_id", "order_date", "order_status"]
].copy()

order_dates["order_date"] = pd.to_datetime(
    order_dates["order_date"]
)

# Keep delivered orders for this evaluation
order_dates = order_dates[
    order_dates["order_status"] == "Delivered"
].copy()

# One row per order
order_dates = order_dates.drop_duplicates(
    subset="order_id"
).sort_values("order_date")

# Choose the cutoff at 80% of the timeline's orders
cutoff_index = int(len(order_dates) * 0.8)
cutoff_date = order_dates.iloc[cutoff_index]["order_date"]

train_orders = set(
    order_dates.loc[
        order_dates["order_date"] < cutoff_date,
        "order_id"
    ]
)

test_orders = set(
    order_dates.loc[
        order_dates["order_date"] >= cutoff_date,
        "order_id"
    ]
)

print("Cutoff date:", cutoff_date)
print("Training orders:", len(train_orders))
print("Testing orders:", len(test_orders))
print("Overlap:", len(train_orders & test_orders))

Cutoff date: 2026-03-16 00:00:00
Training orders: 18886
Testing orders: 4743
Overlap: 0


In [60]:
train_transaction_data = transaction_data[
    transaction_data["order_id"].isin(train_orders)
].copy()

train_transactions = (
    train_transaction_data
    .groupby("order_id")["product_name"]
    .apply(list)
    .tolist()
)

print("Training transactions:", len(train_transactions))
print("Expected training orders:", len(train_orders))

Training transactions: 18886
Expected training orders: 18886


In [64]:
from mlxtend.preprocessing import TransactionEncoder

encoder_train = TransactionEncoder()
train_array = encoder_train.fit_transform(train_transactions)

train_basket = pd.DataFrame(
    train_array,
    columns=encoder_train.columns_
)

print(train_basket.shape)

(18886, 208)


In [65]:
from mlxtend.frequent_patterns import fpgrowth

train_itemsets = fpgrowth(
    train_basket,
    min_support=0.002,
    use_colnames=True
)

print("Frequent itemsets:", len(train_itemsets))

Frequent itemsets: 306


In [ ]:
from mlxtend.frequent_patterns import association_rules

train_rules = association_rules(
    train_itemsets,
    metric="confidence",
    min_threshold=0.1
)

useful_train_rules = train_rules[
    (train_rules["confidence"] >= 0.20) &
    (train_rules["lift"] > 2) &
    (train_rules["support"] >= 0.002)
].copy()

useful_train_rules = useful_train_rules.sort_values(
    "lift", ascending=False
)

print("All training rules:", len(train_rules))
print("Filtered training rules:", len(useful_train_rules))

useful_train_rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(15)

In [ ]:
# Separate training and testing purchase records
train_data = transaction_data[
    transaction_data["order_id"].isin(train_orders)
].copy()

test_data = transaction_data[
    transaction_data["order_id"].isin(test_orders)
].copy()


train_customer_history = (
    train_data.groupby("customer_id")["product_name"]
    .apply(set)
    .to_dict()
)

test_customer_purchases = (
    test_data.groupby("customer_id")["product_name"]
    .apply(set)
    .to_dict()
)

popular_products_train = (
    train_basket.sum()
    .sort_values(ascending=False)
)

print("Customers with training purchases:",
      len(train_customer_history))

print("Customers with test purchases:",
      len(test_customer_purchases))

print("Training records:", len(train_data))
print("Testing records:", len(test_data))

Customers with training purchases: 4003
Customers with test purchases: 2545
Training records: 49367
Testing records: 12450


In [68]:
from sklearn.metrics import precision_score

# Customers with both past history and future purchases
evaluation_customers = (
    set(train_customer_history)
    & set(test_customer_purchases)
)

evaluation_results = []

for customer_id in evaluation_customers:
    purchased_before = train_customer_history[customer_id]
    actual_future = test_customer_purchases[customer_id]

    recs = recommend_products(
        purchased_products=purchased_before,
        rules_df=useful_train_rules,
        popular_products=popular_products_train,
        top_n=5
    )

    recommended = set(recs["recommended_product"])

    hits = recommended & actual_future

    precision_at_5 = len(hits) / 5
    recall_at_5 = (
        len(hits) / len(actual_future)
        if actual_future else 0
    )

    evaluation_results.append({
        "customer_id": customer_id,
        "recommendations": list(recommended),
        "actual_future": list(actual_future),
        "hits": len(hits),
        "precision_at_5": precision_at_5,
        "recall_at_5": recall_at_5
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Customers evaluated:", len(evaluation_df))
print("Mean Precision@5:", evaluation_df["precision_at_5"].mean())
print("Mean Recall@5:", evaluation_df["recall_at_5"].mean())
print("Customers with at least one hit:",
      (evaluation_df["hits"] > 0).sum())

Customers evaluated: 2148
Mean Precision@5: 0.036312849162011177
Mean Recall@5: 0.03869703715434642
Customers with at least one hit: 331
